# MyDigitalTwin — Twitter/X
**Notebook 04 — Ingestion, exploration, nettoyage → Delta Lake**

Sources :
- `data/processed/X/data/tweets.js` → tes tweets
- `data/processed/X/data/like.js` → tweets likés

Outputs :
- `warehouse/twitter_tweets`
- `warehouse/twitter_likes`

## Objectifs
- **Clone NLP** : corpus de tes tweets pour TF-IDF / N-grams
- **K-Means** : activité temporelle

## 0. Initialisation Spark

In [1]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json, re

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Twitter") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

TW_ROOT     = os.path.join(RAW_DATA, "X", "data")

def load_twitter_js(path):
    """Charge un fichier .js Twitter en retirant l'assignation window.YTD."""
    with open(path, encoding="utf-8", errors="replace") as f:
        content = f.read()
    clean = re.sub(r'^window\.YTD\.\w+\.\w+\s*=\s*', '', content.strip())
    return json.loads(clean)

Spark version : 3.5.5


26/04/27 13:05:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE 1 — Tweets
### 1.1 Ingestion

In [2]:
raw_tweets = load_twitter_js(f"{TW_ROOT}/tweets.js")
print(f"Tweets bruts : {len(raw_tweets):,}")

def parse_tweets(raw):
    rows = []
    for entry in raw:
        t = entry.get("tweet", entry)

        text      = t.get("full_text", t.get("text", ""))
        ts        = t.get("created_at", "")
        tweet_id  = t.get("id_str", "")
        lang      = t.get("lang", "")
        rt_count  = int(t.get("retweet_count", 0) or 0)
        fav_count = int(t.get("favorite_count", 0) or 0)
        is_rt     = text.startswith("RT @")
        is_reply  = bool(t.get("in_reply_to_status_id_str"))

        # Nettoyer le texte : retirer les mentions au début des replies
        clean_text = re.sub(r'^(@\w+\s*)+', '', text).strip()

        # Extraire les URLs
        urls = [u.get("expanded_url", "") 
                for u in t.get("entities", {}).get("urls", [])]

        # Extraire les hashtags
        hashtags = [h.get("text", "").lower() 
                    for h in t.get("entities", {}).get("hashtags", [])]

        # Extraire les mentions
        mentions = [m.get("screen_name", "") 
                    for m in t.get("entities", {}).get("user_mentions", [])]

        rows.append({
            "tweet_id":     tweet_id,
            "text":         clean_text,
            "raw_text":     text,
            "lang":         lang,
            "created_at":   ts,
            "retweet_count": rt_count,
            "favorite_count": fav_count,
            "is_retweet":   is_rt,
            "is_reply":     is_reply,
            "has_media":    bool(t.get("entities", {}).get("media")),
            "hashtags":     " ".join(hashtags),
            "mentions":     " ".join(mentions),
            "urls":         " ".join(urls),
            "char_count":   len(clean_text),
            "word_count":   len(clean_text.split()),
        })
    return rows

tweet_rows = parse_tweets(raw_tweets)
print(f"Tweets parsés : {len(tweet_rows):,}")
print("Exemple :", {k: v for k, v in tweet_rows[0].items() if k in ['text', 'created_at', 'lang']})

Tweets bruts : 321
Tweets parsés : 321
Exemple : {'text': 'RT @ohimetenshi: Here we go https://t.co/Nmx6PVFlCF', 'lang': 'en', 'created_at': 'Sun Apr 19 16:41:53 +0000 2026'}


In [3]:
from datetime import datetime, timezone

def parse_twitter_date(date_str):
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.timestamp() * 1000
    except:
        return None

# 1. D'abord remplir timestamp_ms dans les rows Python
for row in tweet_rows:
    ts_ms = parse_twitter_date(row["created_at"])
    row["timestamp_ms"] = int(ts_ms) if ts_ms else 0

# 2. Ensuite créer le DataFrame
schema_tweets = StructType([
    StructField("tweet_id",        StringType(),  True),
    StructField("text",            StringType(),  True),
    StructField("raw_text",        StringType(),  True),
    StructField("lang",            StringType(),  True),
    StructField("created_at",      StringType(),  True),
    StructField("retweet_count",   IntegerType(), True),
    StructField("favorite_count",  IntegerType(), True),
    StructField("is_retweet",      BooleanType(), True),
    StructField("is_reply",        BooleanType(), True),
    StructField("has_media",       BooleanType(), True),
    StructField("hashtags",        StringType(),  True),
    StructField("mentions",        StringType(),  True),
    StructField("urls",            StringType(),  True),
    StructField("char_count",      IntegerType(), True),
    StructField("word_count",      IntegerType(), True),
    StructField("timestamp_ms",    LongType(),    True),
])

df_tweets = spark.createDataFrame(tweet_rows, schema=schema_tweets)

# 3. Ensuite les transformations Spark
df_tweets = df_tweets.withColumn(
    "event_date", F.to_timestamp(F.col("timestamp_ms") / 1000)
)

df_tweets = df_tweets \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("twitter"))

print(f"Lignes : {df_tweets.count():,}")
df_tweets.select("text", "event_date", "lang", "is_retweet", "is_reply").show(5, truncate=60)

Lignes : 321
+------------------------------------------------------------+-------------------+----+----------+--------+
|                                                        text|         event_date|lang|is_retweet|is_reply|
+------------------------------------------------------------+-------------------+----+----------+--------+
|         RT @ohimetenshi: Here we go https://t.co/Nmx6PVFlCF|2026-04-19 16:41:53|  en|      true|   false|
|RT @encoreksg: le match d’une vie 🙏🏾 https://t.co/x0kE1...|2026-04-19 16:36:34|  fr|      true|   false|
|RT @Papino_sock: La CAF a supprimé cette vidéo maximum de...|2026-03-19 22:56:32|  fr|      true|   false|
|RT @Arsenal_rep1: 🚨🎙️| Neymar Jr: 🗣️\n\n"I watched som...|2026-03-19 22:54:57|  en|      true|   false|
|RT @CanalplusFoot: MAX DOWMAN DANS L'HISTOIRE DE LA PREMI...|2026-03-15 12:05:09|  fr|      true|   false|
+------------------------------------------------------------+-------------------+----+----------+--------+
only showing top 5 r

### 1.2 Exploration

In [4]:
print("=== Répartition des types de tweets ===")
df_tweets.groupBy("is_retweet", "is_reply") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("\n=== Langues utilisées ===")
df_tweets.groupBy("lang").count().orderBy(F.desc("count")).show()

print("\n=== Tweets par année ===")
df_tweets.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_tweets.groupBy("event_hour").count().orderBy("event_hour").show()

=== Répartition des types de tweets ===


+----------+--------+-----+
|is_retweet|is_reply|count|
+----------+--------+-----+
|     false|    true|  180|
|      true|   false|  139|
|     false|   false|    2|
+----------+--------+-----+


=== Langues utilisées ===
+----+-----+
|lang|count|
+----+-----+
|  fr|  178|
|  en|   59|
| qme|   39|
| und|   10|
|  es|    8|
| qam|    5|
| zxx|    2|
|  tl|    2|
|  in|    2|
|  cs|    2|
|  de|    2|
|  pt|    2|
|  ht|    2|
|  lt|    2|
|  tr|    1|
|  nl|    1|
|  ro|    1|
|  sl|    1|
|  it|    1|
|  no|    1|
+----+-----+


=== Tweets par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2019|   51|
|      2020|  105|
|      2021|   39|
|      2022|   16|
|      2023|    8|
|      2024|   31|
|      2025|   54|
|      2026|   17|
+----------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|    9|
|         1|    3|
|         2|    3|
|         4|    2|
|         5|    8|
|         6|    8|
|        

In [5]:
print("=== Tes tweets les plus likés ===")
df_tweets.filter(~F.col("is_retweet")) \
    .orderBy(F.desc("favorite_count")) \
    .select("text", "favorite_count", "retweet_count", "event_date") \
    .limit(10) \
    .show(truncate=70)

print("\n=== Top hashtags ===")
df_tweets.filter(F.col("hashtags") != "") \
    .withColumn("hashtag", F.explode(F.split("hashtags", " "))) \
    .filter(F.col("hashtag") != "") \
    .groupBy("hashtag") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show()

print("\n=== Top 20 mots (hors retweets) ===")
df_tweets.filter(~F.col("is_retweet")) \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .filter(~F.col("word").startswith("@")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

print("\n=== Stats texte (hors retweets) ===")
df_tweets.filter(~F.col("is_retweet")) \
    .agg(
        F.avg("char_count").alias("avg_chars"),
        F.avg("word_count").alias("avg_words"),
        F.max("char_count").alias("max_chars"),
    ).show()

=== Tes tweets les plus likés ===
+----------------------------------------------------------------------+--------------+-------------+-------------------+
|                                                                  text|favorite_count|retweet_count|         event_date|
+----------------------------------------------------------------------+--------------+-------------+-------------------+
|                                               https://t.co/GYi9f2TZbt|           106|            0|2025-12-08 16:42:21|
|                                               https://t.co/aYtJC3Gzms|           103|            3|2021-07-08 20:28:48|
|                                                  Il reste Paris Plage|            48|            1|2020-08-14 13:00:48|
|                                               https://t.co/Oxf19MOuIQ|            22|            0|2022-04-15 05:11:20|
|                                               https://t.co/eaVDMFlvjz|            12|            0|2020-08-15 

### 1.3 Écriture Parquet

In [6]:
df_tweets.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "twitter_tweets"))
print(f"twitter_tweets -- {df_tweets.count():,} lignes")

26/04/27 13:06:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


twitter_tweets -- 321 lignes


---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [7]:
raw_likes = load_twitter_js(f"{TW_ROOT}/like.js")
print(f"Likes bruts : {len(raw_likes):,}")

def parse_likes(raw):
    rows = []
    for entry in raw:
        lk = entry.get("like", entry)
        rows.append({
            "tweet_id":    lk.get("tweetId", ""),
            "full_text":   lk.get("fullText", ""),
            "post_url":    lk.get("expandedUrl", ""),
        })
    return rows

like_rows = parse_likes(raw_likes)
print(f"Likes parsés : {len(like_rows):,}")
print("Exemple :", like_rows[0])

Likes bruts : 70,136
Likes parsés : 70,136
Exemple : {'tweet_id': '2047733698678690013', 'full_text': 'AI-powered CAD that actually WORKS is the final frontier for LLMs.\n\nImagine vibe-coding, but vibe-manufacturing. https://t.co/hT4wxKtIOO', 'post_url': 'https://twitter.com/i/web/status/2047733698678690013'}


In [8]:
schema_likes = StructType([
    StructField("tweet_id",  StringType(), True),
    StructField("full_text", StringType(), True),
    StructField("post_url",  StringType(), True),
])

df_likes = spark.createDataFrame(like_rows, schema=schema_likes)

df_likes = df_likes \
    .withColumn("platform",    F.lit("twitter")) \
    .withColumn("action_type", F.lit("like")) \
    .withColumn("char_count",  F.length("full_text")) \
    .withColumn("word_count",  F.size(F.split(F.trim("full_text"), r"\s+")))

print(f"Lignes : {df_likes.count():,}")
df_likes.show(5, truncate=70)

Lignes : 70,136
+-------------------+-------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------+--------+-----------+----------+----------+
|           tweet_id|                                                                                                                full_text|                                            post_url|platform|action_type|char_count|word_count|
+-------------------+-------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------+--------+-----------+----------+----------+
|2047733698678690013|                                                   AI-powered CAD that actually WORKS is the final frontier for LLMs.\...|https://twitter.com/i/web/status/2047733698678690013| twitter|       like|       136|        16|
|2048008719833673819|   

### 2.2 Exploration

In [9]:
print("=== Nombre total de likes ===")
print(f"  {df_likes.count():,} tweets likés")

print("\n=== Top 20 mots dans les tweets likés ===")
df_likes.filter(F.col("full_text") != "") \
    .withColumn("word", F.explode(F.split(F.lower("full_text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .filter(~F.col("word").startswith("@")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

print("\n=== Exemples de tweets likés ===")
df_likes.select("full_text", "post_url") \
    .limit(10) \
    .show(truncate=80)

=== Nombre total de likes ===
  70,136 tweets likés

=== Top 20 mots dans les tweets likés ===


+-----------+-----+
|       word|count|
+-----------+-----+
|        les| 9898|
|       this| 9531|
|        pas| 9050|
|        que| 8350|
|       pour| 6847|
|      c’est| 6765|
|        des| 6525|
|       post| 6339|
|{learnmore}| 6256|
|        est| 6200|
|        qui| 6065|
|        une| 5545|
|       mais| 4695|
|        sur| 4415|
|       dans| 4331|
|       from| 4298|
|       view| 4200|
|  suspended| 4075|
|   account.| 4073|
|      c'est| 3574|
+-----------+-----+


=== Exemples de tweets likés ===
+------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------+
|                                                                                                                                 full_text|                                            post_url|
+------------------------------------------------------------------------------------------------

### 2.3 Écriture Parquet

In [10]:
df_likes.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "twitter_likes"))
print(f"twitter_likes -- {df_likes.count():,} lignes")

twitter_likes -- 70,136 lignes


In [11]:
# -- PARTIE 3 - Recherches sauvegardees --
saved_search_path = os.path.join(TW_ROOT, "saved-search.js")
saved_rows = []

if os.path.exists(saved_search_path):
    raw_ss = load_twitter_js(saved_search_path)
    for item in raw_ss:
        if isinstance(item, dict):
            query = (item.get("savedSearch", {}) or {}).get("query", "") or item.get("query", "")
        else:
            query = str(item)
        if query:
            saved_rows.append({"query": query, "platform": "twitter"})
    print(f"Recherches sauvegardees : {len(saved_rows):,}")
else:
    print("saved-search.js introuvable -- skipped")

if saved_rows:
    from pyspark.sql.types import StructType, StructField, StringType
    schema_ss = StructType([StructField("query", StringType(), True), StructField("platform", StringType(), True)])
    df_saved_searches = spark.createDataFrame(saved_rows, schema=schema_ss)
    df_saved_searches.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "twitter_saved_searches"))
    print(f"twitter_saved_searches -- {df_saved_searches.count():,} lignes")
    df_saved_searches.show(truncate=60)
else:
    print("Aucune recherche sauvegardee")

Recherches sauvegardees : 0
Aucune recherche sauvegardee


In [12]:
spark.stop()